---
# NLI Fine-Tuning
> Laura Komorek
---

## Fine-Tuning RoBERTa-Base on our Prediction Data

This notebook fine-tunes RoBERTA-base and runs experiments with different types of masking strategies and input representation on our 3 datasets.

Our goals are:
- Load our prediction files for each dataset
- Test different masking strategies
- Test different input representations
- Fine-tune RoBERTa-Base

---

## Imports and Configuration

In [ ]:
import json
import os
from copy import deepcopy
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
from itertools import product

# -- configuration
model_name = "textattack/roberta-base-mnli"
prediction_files=[
    "ontonotes_predictions.json",
    "figer_predictions.json",
    "ultrafine_predictions.json"
]
output_dir= "./Experiments"
batch_size = 16
epochs = 3
lr = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---

## Masking Strategies and Input Representations

### Masking Strategies
We apply masking strategies to the text to modify it before running it through the model.
For example:
- "none": there is no masking here and we are using the text as it is.
- "mask_verbs": we replace common verbs like "is" or "are" with "[MASK]".
- "mask_negation": we replace common negation words with "[MASK]" to see how the model works without "not" etc.
- "mask_stopwords": we are replacing common stopwords wiht "[MASK]" to see how important grammar is

### Input Representation
For input representation we define how our premises and hypotheses are combined as a single input string.
For example:
- "premise_sep_hypothesis": we concatenate premise and hypothesis separated by "[SEP]".
- "hypothesis_sep_premise": we concatenate hypothesis first and then premise, again separated by "[SEP]"


In [ ]:
# -- Masking Strategies (examples)
masking_strategies = {
    "none": lambda text: text,
    "mask_verbs": lambda text: text.replace("is", "[MASK]").replace("are", "[MASK]"),
    "mask_negation": lambda text: text.replace(" not ", " [MASK] ").replace(" no ", " [MASK] ").replace(" never ", " [MASK] "),
    "mask_stopwords": lambda text: " ".join(
        ["[MASK]" if w.lower() in {"the", "a", "an", "is", "are", "in", "on", "at"} else w for w in text.split()]
    )
}


# -- Input Representation (examples)
input_rep = {
    "premise_sep_hypothesis": lambda d: f"{d['premise']} [SEP] {d['hypothesis']}",
    "hypothesis_sep_premise": lambda d: f"{d['hypothesis']} [SEP] {d['premise']}"
}


# -- Metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

---

## Load Data and Run Experiments

We load the prediction JSON files and prepare all the combinations of masking strategies and input representations for experiments.

In [13]:
for pred_file in prediction_files:
    with open(pred_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    dataset_name = os.path.splitext(os.path.basename(pred_file))[0]

    experiment_combos = list(product(masking_strategies.items(), input_rep.items()))

    for (mask_name, mask_fn), (repr_name, repr_fn) in experiment_combos:
        exp_name = f"{dataset_name}__{mask_name}__{repr_name}"
        print(f"\nRunning Experiment: {exp_name}")

        # -- prepare datasets
        texts = [repr_fn({'premise': mask_fn(d['premise']), 'hypothesis': mask_fn(d['hypothesis'])}) for d in data]
        labels = [0 if d['gold_label'] == "NOT ENTAILMENT" else 1 for d in data]
        dataset = Dataset.from_dict({"text": texts, "label": labels})

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            ignore_mismatched_sizes=True
        )
        model.to(device)

        def tokenize_fn(batch):
            return tokenizer(
                batch["text"], 
                padding="max_length", 
                truncation=True,
                max_length=128)

        dataset = dataset.map(tokenize_fn, batched=True)

        # -- TrainingArguments
        exp_output_dir = os.path.join(output_dir, exp_name)
        os.makedirs(exp_output_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=exp_output_dir,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=epochs,
            learning_rate=lr,
            logging_dir=os.path.join(exp_output_dir, "logs"),
            logging_steps=100,
            save_steps=500
        )

        # -- Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
            eval_dataset=dataset,
            compute_metrics=compute_metrics
        )

        # -- start the training
        trainer.train()

        # -- save the best model
        trainer.save_model()

        print(f"\nFinished Experiment: {exp_name}\n")



Running Experiment: ontonotes_predictions__none__premise_sep_hypothesis


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28825.35it/s]
RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-mnli
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/

Step,Training Loss


KeyboardInterrupt: 